# Track 2: Cardiology Medical Expert Fine-Tuning
This notebook demonstrates domain-specific fine-tuning for Cardiology clinical QA using **Unsloth** and **QLoRA** on the `lmassaron/medical-cardiology-qa` dataset.

### Why Unsloth?
- Up to 5x faster training speeds.
- Up to 60% memory savings, allowing larger models (or larger batch sizes) on a single 16GB VRAM GPU.
- Keeps native model quality without performance degradation.

In [ ]:
# Install extra dependencies if needed
# %pip install -U unsloth trl peft bitsandbytes datasets


In [ ]:
import os
import torch
import numpy as np
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

max_seq_length = 1024
dtype = None  # None for auto detection (Float16/Bfloat16 based on hardware)
load_in_4bit = True  # NF4 quantization for memory savings

# Check device capabilities
compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    else torch.float16
)

## 1. Load Model & Setup LoRA
We load the Unsloth-optimized **Phi-4 Mini Instruct** model (3.8B parameters) and inject PEFT adapters.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-4-mini-instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Setup LoRA adapters targeting all attention and MLP projections
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,  # Unsloth optimized to 0
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 2. Load and Prepare the Cardiology QA Dataset
We load the cardiology QA dataset, split it into training/validation, and map the chat messages using the model's chat template.

In [ ]:
DATASET_ID = "lmassaron/medical-cardiology-qa"
print(f"Loading {DATASET_ID}...")
dataset = load_dataset(DATASET_ID, split="train")

# Sample evaluation split (10%) reproducible seed
n = len(dataset)
rng = np.random.default_rng(42)
all_idx = rng.permutation(n)
cut = int(n * 0.9)
train_idx, eval_idx = all_idx[:cut], all_idx[cut:]

train_ds = dataset.select(train_idx)
eval_ds = dataset.select(eval_idx)
print(f"Train samples: {len(train_ds)} | Eval samples: {len(eval_ds)}")

# Let's print a sample structure
print("Sample message sequence:", train_ds[0]["messages"])

## 3. Formatting Prompts
We map the dataset using the conversation structure.

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}


train_mapped = train_ds.map(formatting_prompts_func, batched=True)
eval_mapped = eval_ds.map(formatting_prompts_func, batched=True)
print("Prompt Preview:\n", train_mapped[0]["text"])

## 4. Run SFT Trainer using Unsloth
We configure the SFTTrainer using Unsloth's optimized training engine.

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_mapped,
    eval_dataset=eval_mapped,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=0,
        max_steps=100,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_available() or torch.cuda.get_device_capability()[0] < 8,
        bf16=torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8,
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=50,
        save_steps=50,
        output_dir="unsloth-medical-model",
        report_to="none",
        dataset_kwargs={
            "add_special_tokens": False,
        },
    ),
)

# Disable KV cache during training to save memory
model.config.use_cache = False

trainer_stats = trainer.train()

## 5. Save the Adapter
We save the fine-tuned adapter weights to disk.

In [ ]:
model.save_pretrained("unsloth-medical-adapter")
tokenizer.save_pretrained("unsloth-medical-adapter")
print("Adapter successfully saved!")

## 6. Evaluation and Inference
We put the model into Unsloth's optimized inference mode and run evaluations on our held-out test split. We calculate the average **Perplexity** on the cardiology reference answers (demonstrating how 'surprised' the model is by the clinical ground truth) and print a few example outputs for comparison.

In [ ]:
import pandas as pd
from tqdm import tqdm

FastLanguageModel.for_inference(model)  # 2x faster inference

eval_results = []
# We evaluate on a sample of 20 evaluation conversations to run quickly
num_eval_samples = min(20, len(eval_ds))
print(
    f"Calculating perplexity and generating answers for {num_eval_samples} evaluation examples..."
)

for i in tqdm(range(num_eval_samples)):
    messages = eval_ds[i]["messages"]
    user_question = next(m["content"] for m in messages if m["role"] == "user")
    expected_answer = next(m["content"] for m in messages if m["role"] == "assistant")

    # Format inputs for model generation
    encoded = tokenizer.apply_chat_template(
        messages[:-1], tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    # Generate answer
    with torch.no_grad():
        outputs = model.generate(
            input_ids=encoded,
            max_new_tokens=256,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(
        outputs[0][encoded.shape[1] :], skip_special_tokens=True
    ).strip()

    # Calculate perplexity over target answer tokens
    ref_ids = tokenizer(
        expected_answer, return_tensors="pt", add_special_tokens=False
    ).input_ids.to("cuda")
    full_ids = torch.cat([encoded, ref_ids], dim=1)
    labels = full_ids.clone()
    labels[:, : encoded.shape[1]] = -100  # Mask out prompt tokens

    with torch.no_grad():
        outputs_loss = model(full_ids, labels=labels)
        loss = outputs_loss.loss
        perplexity = torch.exp(loss).item()

    eval_results.append(
        {
            "question": user_question,
            "expected": expected_answer,
            "generated": generated,
            "perplexity": perplexity,
        }
    )

eval_df = pd.DataFrame(eval_results)
print(f"\nAverage Evaluation Perplexity: {eval_df['perplexity'].mean():.4f}")

# Print a sample evaluation comparison
print("\n--- Sample Comparison ---")
print("Question:", eval_df.iloc[0]["question"])
print("\nExpected Answer:", eval_df.iloc[0]["expected"])
print("\nGenerated Answer:", eval_df.iloc[0]["generated"])
print(f"Perplexity: {eval_df.iloc[0]['perplexity']:.4f}")